# 2 — Boost PFC in DCM (Discontinuous Conduction Mode)

> **Goal.** Derive the DCM analytical model, show that the input
> current is *naturally* proportional to $v_g(t)$ if $D$ is held
> approximately constant over a line half-cycle, design a single
> voltage-loop PI compensator, and verify with a switched closed-loop
> simulation that we get PF > 0.99 with regulated $V_o$.

**Prerequisites**

- Boost modeling notebook (`projects/converters/boost/`)
- PFC basics notebook (`01_boost_pfc_basics.ipynb`)

**What you'll be able to do at the end**

1. Sketch the 3-phase $i_L$ waveform in DCM (charge → discharge → zero)
   and derive the average input current $\langle i_g \rangle_{T_{sw}}$.
2. Compute the equivalent resistance $R_e = 2 L f_{sw}/D^2$ and use it
   to derive the steady-state duty $D(V_{ac})$.
3. Apply the cusp-distortion correction $F(m_{pk})$ to get the *exact*
   duty for a target $V_o$.
4. Derive the line-cycle averaged voltage-loop plant and design a
   PI compensator with $f_c \le 2 f_{line}/10$.
5. Discretize and run the switched closed-loop sim, then measure PF
   and THD and check they meet the IEC 61000-3-2 target.


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

import numpy as np
from scipy import signal
import matplotlib.pyplot as plt

from boost_pfc_model import (
    BoostPFCParams,
    dcm_critical_inductance, dcm_equivalent_resistance,
    dcm_steady_state_duty, dcm_steady_state_duty_exact,
    dcm_power_correction_factor,
    dcm_input_current_instantaneous,
    dcm_voltage_loop_plant,
    simulate_open_loop_dcm, simulate_closed_loop_dcm,
    operating_point_report,
    power_factor, thd_current, line_band_filter,
)

plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

p = BoostPFCParams.dcm_design()
print(operating_point_report(p, mode="DCM"))


## 1. The 3-phase DCM cycle

Within one $T_{sw}$, three phases:

```
phase 1 (S on):     L · di_L/dt = v_g                  i_L rises from 0 to i_pk
phase 2 (S off, D on): L · di_L/dt = v_g - V_o         i_L falls from i_pk to 0
phase 3 (S off, D off): L · di_L/dt = 0                i_L = 0  (DCM "dead" phase)
```

With on-duty $D$ and off-duty $D_2 = D \cdot v_g/(V_o - v_g)$ (from
volt-second balance on the inductor):

$$
i_{pk}(t) = v_g(t) \cdot D \cdot T_{sw} / L
$$

The **average** $i_g$ over one $T_{sw}$ is half the triangle area
over the period:

$$
\langle i_g \rangle_{T_{sw}}
= \frac{1}{2} \cdot i_{pk} \cdot (D + D_2)
= \frac{v_g}{R_e} \cdot \frac{1}{1 - v_g/V_o}
$$

where

$$
\boxed{R_e = \frac{2 L f_{sw}}{D^2}}
$$

is the **equivalent input resistance** of the DCM boost — Erickson's
key insight. With $D$ constant over a line cycle, the input looks
like a resistor $R_e$ in parallel with a small cusp-distortion term.


## 2. The cusp distortion

The factor $1/(1 - v_g/V_o)$ in the average input current grows as
$v_g$ approaches $V_o$. At low line ($V_{g,pk} \ll V_o$), the
distortion is small and $\langle i_g \rangle \approx v_g/R_e$ is
nearly resistive — natural PFC works beautifully.

At high line ($V_{g,pk}$ close to $V_o$), the distortion grows
rapidly and PF degrades. This is the **fundamental limitation** of
fixed-$D$ DCM PFC for universal-input designs.


In [ ]:
# Plot the input current shape over a line cycle at several V_ac values
fig, ax = plt.subplots(figsize=(10, 4.5))
t_line = np.linspace(0, 1/p.f_line, 2000)
for V_ac, color in [(90, "C0"), (120, "C1"), (180, "C2"), (230, "C3"), (265, "C4")]:
    D = dcm_steady_state_duty_exact(p, V_ac=V_ac)
    i_g = dcm_input_current_instantaneous(p, t_line, D, V_ac=V_ac)
    ax.plot(t_line*1000, i_g, color=color, label=f"V_ac = {V_ac} V (D={D:.3f})")

ax.set_xlabel("Time [ms]"); ax.set_ylabel("$\\langle i_g \\rangle_{T_{sw}}$ [A]")
ax.set_title("DCM PFC input current envelope — cusp distortion grows with V_ac")
ax.legend()
plt.tight_layout(); plt.show()


In [ ]:
# The cusp-distortion correction factor F(m_pk)
m_range = np.linspace(0.0, 0.98, 200)
F_vals = np.array([dcm_power_correction_factor(m) for m in m_range])
fig, ax = plt.subplots(figsize=(10, 4))
ax.semilogy(m_range, F_vals, "C0", linewidth=2)
ax.axhline(1.0, color="k", linestyle=":", alpha=0.4, label="resistive limit (F=1)")
for V_ac, color in [(90, "C1"), (120, "C2"), (180, "C3"), (230, "C4"), (265, "C5")]:
    m = np.sqrt(2)*V_ac/p.V_o
    F = dcm_power_correction_factor(m)
    ax.plot(m, F, "o", color=color, label=f"V_ac={V_ac}V (m={m:.2f}, F={F:.2f})")
ax.set_xlabel("$m_{pk} = V_{g,pk}/V_o$")
ax.set_ylabel("Correction factor $F(m_{pk})$")
ax.set_title("DCM PFC cusp-distortion correction factor (Erickson)")
ax.legend(fontsize=8, loc="upper left"); plt.tight_layout(); plt.show()


## 3. Steady-state duty — linearized vs exact

The Erickson-textbook "linearized" duty assumes $m_{pk} \to 0$ and
the converter is a pure resistor at $R_e$:

$$
D_{lin} = \frac{\sqrt{2 L f_{sw} P_o}}{V_{ac,rms}}
$$

The **exact** duty, including cusp distortion, solves

$$
P_o = \frac{V_{ac,rms}^2 \, D^2}{2 L f_{sw}} \cdot F(m_{pk})
$$

giving

$$
\boxed{D_{exact} = \sqrt{\frac{2 L f_{sw} P_o}{F(m_{pk}) \, V_{ac,rms}^2}}}
$$

For universal mains, the two differ by 5-40%. Use $D_{exact}$ when
setting up the open-loop verification; in closed-loop the controller
finds the right duty automatically.


In [ ]:
print(f"{'V_ac':>6} {'D_lin':>8} {'D_exact':>8} {'F(m_pk)':>10}")
for V_ac in [90, 120, 180, 230, 265]:
    D_lin = dcm_steady_state_duty(p, V_ac=V_ac)
    D_ex = dcm_steady_state_duty_exact(p, V_ac=V_ac)
    m = np.sqrt(2)*V_ac/p.V_o
    F = dcm_power_correction_factor(m)
    print(f"{V_ac:6.0f} {D_lin:8.4f} {D_ex:8.4f} {F:10.3f}")


## 4. Open-loop natural PFC — proof of concept

Run the switched sim with a **fixed** $D$ (no controller) and watch
the input current naturally take the shape of $v_{ac}(t)$. PF should
be > 0.99 at low line.


In [ ]:
sim_ol = simulate_open_loop_dcm(p, V_ac=120.0, n_line_cycles=8, samples_per_period=80)

fig, axs = plt.subplots(3, 1, figsize=(11, 9), sharex=True)
axs[0].plot(sim_ol['t']*1000, sim_ol['v_o'], "C2", linewidth=1.0)
axs[0].axhline(p.V_o, color="k", linestyle=":", alpha=0.4, label=f"target {p.V_o} V")
axs[0].set_ylabel("$v_o$ [V]")
axs[0].set_title("Open-loop DCM PFC at V_ac = 120 V, fixed D (exact)")
axs[0].legend()

axs[1].plot(sim_ol['t']*1000, sim_ol['v_ac'], "C0", linewidth=1.0, label="$v_{ac}$")
ax2 = axs[1].twinx()
ax2.plot(sim_ol['t']*1000, sim_ol['i_in'], "C3", linewidth=0.6, alpha=0.7, label="$i_{in}$ (raw)")
axs[1].set_ylabel("$v_{ac}$ [V]"); ax2.set_ylabel("$i_{in}$ [A]")
axs[1].set_title("Line voltage and input current (raw, with switching ripple)")
axs[1].legend(loc="upper left"); ax2.legend(loc="upper right")

# Filter the input current to the line band (anti-alias PQ measurement)
fs = 1.0/(sim_ol['t'][1] - sim_ol['t'][0])
i_filt = line_band_filter(sim_ol['i_in'], fs, p.f_line, n_harmonics=20)
axs[2].plot(sim_ol['t']*1000, sim_ol['v_ac']/sim_ol['v_ac'].max(), "C0:", alpha=0.5, label="$v_{ac}$ shape")
axs[2].plot(sim_ol['t']*1000, i_filt/np.max(np.abs(i_filt)), "C3", linewidth=1.5, label="$i_{in}$ filtered (normalized)")
axs[2].set_xlabel("Time [ms]"); axs[2].set_ylabel("Normalized")
axs[2].set_title("Filtered input current vs line voltage — natural PFC")
axs[2].legend()
plt.tight_layout(); plt.show()

# Metrics on the last 4 cycles (after transient)
mask = sim_ol['t'] >= 4.0/p.f_line
pf = power_factor(sim_ol['v_ac'][mask], sim_ol['i_in'][mask], f_s=fs, f_line=p.f_line)
i_filt_m = line_band_filter(sim_ol['i_in'][mask], fs, p.f_line, 20)
thd = thd_current(i_filt_m, fs, p.f_line)
print(f"Open-loop DCM @ V_ac=120V (steady-state): PF = {pf:.4f}, THD = {thd*100:.2f}%")


## 5. Line-cycle averaged voltage-loop plant

Now that we believe in DCM PFC, design the voltage compensator.
Average the converter's power flow over a *line cycle* (slow
dynamics, ignore the switching) — the bulk cap balance becomes:

$$
C V_o \frac{d V_o}{dt} = P_{in}(D, V_{ac}) - \frac{V_o^2}{R_{load}}
$$

With $P_{in} = V_{ac}^2 D^2 / (2 L f_{sw})$ (resistive-emulator
approximation), linearize around $(V_o, D)$:

$$
C V_o \frac{d \hat v_o}{dt} = \frac{V_{ac}^2 D}{L f_{sw}} \hat d
                              - \frac{2 V_o}{R_{load}} \hat v_o
$$

Rearrange:

$$
\boxed{
G_{vd}(s) = \frac{\hat v_o}{\hat d}
= \frac{K_{dc}}{1 + s \tau}, \quad
K_{dc} = \frac{V_{ac}^2 D}{L f_{sw}} \cdot \frac{R_{load}}{2 V_o}, \quad
\tau = \frac{R_{load} C}{2}
}
$$

A clean **first-order** plant! Compare to the boost converter's
second-order-plus-RHP-zero misery — DCM PFC is dramatically easier.


In [ ]:
plant = dcm_voltage_loop_plant(p)
print(f"DCM voltage-loop plant:")
print(f"  num = {plant.num}")
print(f"  den = {plant.den}")
pole_freq = (plant.den[1]/plant.den[0]) / (2*np.pi)
print(f"  pole at f = {pole_freq:.3f} Hz")
print(f"  DC gain   = {plant.num[0]/plant.den[1]:.2f} V/duty")

# Bode plot
f = np.logspace(-1, 3, 500)
w = 2*np.pi*f
_, mag, ph = signal.bode(plant, w=w)
fig, (ax_mag, ax_ph) = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
ax_mag.semilogx(f, mag, "C0", linewidth=2)
ax_mag.axhline(0, color="k", linestyle=":", alpha=0.3)
ax_mag.axvline(2*p.f_line, color="C3", linestyle=":", alpha=0.5, label=f"2·f_line = {2*p.f_line} Hz")
ax_mag.axvline(2*p.f_line/10, color="C2", linestyle=":", alpha=0.5, label=f"BW ceiling ≈ {2*p.f_line/10} Hz")
ax_ph.semilogx(f, ph, "C0", linewidth=2)
ax_ph.axhline(-90, color="k", linestyle=":", alpha=0.3)
ax_mag.set_ylabel("Magnitude [dB]"); ax_ph.set_ylabel("Phase [deg]")
ax_ph.set_xlabel("Frequency [Hz]")
ax_mag.set_title("DCM voltage-loop plant — first-order low-pass")
ax_mag.legend(); plt.tight_layout(); plt.show()


## 6. PI compensator

For a first-order plant, a simple **PI** with the zero placed at the
plant pole and the gain set for the desired crossover is the
textbook recipe. No Type-III needed.

$$
G_c(s) = \frac{K}{s} (1 + s \tau_z)
$$

Place $\tau_z = R_{load} C / 2$ → the zero exactly cancels the plant
pole, giving an open-loop integrator. Then $K$ sets the crossover.

Target: $f_c = 5$ Hz, PM ≈ 90° (achievable because the plant is
single-pole and we've cancelled it).


In [ ]:
V_ramp = 5.0
plant_pwm = signal.TransferFunction([x/V_ramp for x in plant.num], plant.den)

f_c = 10.0
omega_c = 2*np.pi*f_c
tau_z = plant.den[0]/plant.den[1]
# Compensator: K/s · (1 + s·tau_z). At omega_c (after canceling pole):
# open-loop = K/s · plant_dc_gain → magnitude at omega_c is K · plant_dc_gain / omega_c
plant_dc_gain = plant_pwm.num[0]/plant_pwm.den[1]
K = omega_c / plant_dc_gain
print(f"PI: K = {K:.4g}, tau_z = {tau_z:.4f} s ({1/(2*np.pi*tau_z):.2f} Hz zero)")

Gc = signal.TransferFunction([K*tau_z, K], [1.0, 0.0])
T_open = signal.TransferFunction(np.polymul(Gc.num, plant_pwm.num),
                                   np.polymul(Gc.den, plant_pwm.den))
_, mag_T, ph_T = signal.bode(T_open, w=2*np.pi*f)
idx_cross = np.argmin(np.abs(mag_T))
f_cross = f[idx_cross]
pm = 180 + ph_T[idx_cross]

fig, (ax_mag, ax_ph) = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
ax_mag.semilogx(f, mag_T, "C3", linewidth=2)
ax_mag.axhline(0, color="k", linestyle=":")
ax_mag.axvline(f_cross, color="g", linestyle=":", alpha=0.5, label=f"$f_c$ = {f_cross:.2f} Hz")
ax_mag.axvline(2*p.f_line, color="C3", linestyle=":", alpha=0.5, label=f"2·f_line = {2*p.f_line} Hz")
ax_ph.semilogx(f, ph_T, "C3", linewidth=2)
ax_ph.axhline(-180, color="k", linestyle=":")
ax_ph.axhline(-90, color="C2", linestyle=":", alpha=0.4, label="-90° (integrator)")
ax_mag.set_ylabel("Magnitude [dB]"); ax_ph.set_ylabel("Phase [deg]")
ax_ph.set_xlabel("Frequency [Hz]")
ax_mag.set_title(f"DCM voltage loop: f_c = {f_cross:.2f} Hz, PM = {pm:.1f}°")
ax_mag.legend(); ax_ph.legend(); plt.tight_layout(); plt.show()
print(f"f_c achieved = {f_cross:.2f} Hz, PM = {pm:.1f}°")


## 7. Discretization

Tustin (bilinear) at $T_s = 1/f_{sw}$:


In [ ]:
T_s = p.T_sw
bv_raw, av_raw, _ = signal.cont2discrete((Gc.num, Gc.den), dt=T_s, method='bilinear')
bv = np.asarray(bv_raw).flatten() / av_raw[0]
av = np.asarray(av_raw) / av_raw[0]
print(f"Sample period T_s = {T_s*1e6:.3f} µs")
print(f"  b = {bv}")
print(f"  a = {av}")


## 8. Switched closed-loop simulation

Run with $V_{ac}$ = 120 V (nominal-ish in the universal range)
across 10 line cycles. Expect $V_o$ to regulate to 400 V with the
characteristic 100 Hz ripple. PF should be > 0.99, THD modest.


In [ ]:
sim_cl = simulate_closed_loop_dcm(p, bv, av, V_ac=120.0, n_line_cycles=10,
                                    samples_per_period=80, v_ref=400.0,
                                    V_ramp=V_ramp, warm_start=True)

fig, axs = plt.subplots(4, 1, figsize=(12, 11), sharex=True)
axs[0].plot(sim_cl['t']*1000, sim_cl['v_o'], "C2", linewidth=1.0)
axs[0].axhline(400.0, color="k", linestyle=":", alpha=0.4, label="$V_{ref}$ = 400 V")
axs[0].set_ylabel("$v_o$ [V]")
axs[0].set_title(f"Closed-loop DCM PFC @ V_ac = 120 V (V_g_pk = {np.sqrt(2)*120:.0f} V)")
axs[0].legend()

axs[1].plot(sim_cl['t']*1000, sim_cl['v_ac'], "C0", linewidth=1.0)
axs[1].set_ylabel("$v_{ac}$ [V]")

# Filtered input current
fs = 1.0/(sim_cl['t'][1] - sim_cl['t'][0])
i_filt = line_band_filter(sim_cl['i_in'], fs, p.f_line, 20)
axs[2].plot(sim_cl['t']*1000, sim_cl['i_in'], "C3", linewidth=0.4, alpha=0.4, label="raw $i_{in}$")
axs[2].plot(sim_cl['t']*1000, i_filt, "C1", linewidth=1.5, label="$i_{in}$ filtered")
axs[2].set_ylabel("$i_{in}$ [A]"); axs[2].legend(loc="lower right")

axs[3].plot(sim_cl['t']*1000, sim_cl['duty'], "C4", linewidth=0.6)
axs[3].set_ylabel("Duty"); axs[3].set_xlabel("Time [ms]")
plt.tight_layout(); plt.show()

# Steady-state metrics (drop first 5 cycles for transient)
mask = sim_cl['t'] >= 5.0/p.f_line
pf = power_factor(sim_cl['v_ac'][mask], sim_cl['i_in'][mask], f_s=fs, f_line=p.f_line)
i_filt_m = line_band_filter(sim_cl['i_in'][mask], fs, p.f_line, 20)
thd = thd_current(i_filt_m, fs, p.f_line)
v_o_mean = sim_cl['v_o'][mask].mean()
v_o_pp = sim_cl['v_o'][mask].max() - sim_cl['v_o'][mask].min()
print()
print("Closed-loop DCM PFC steady-state metrics:")
print(f"  V_o mean        = {v_o_mean:.2f} V  (target 400.0 V, error {(v_o_mean-400)/400*100:.2f}%)")
print(f"  V_o ripple pp   = {v_o_pp:.2f} V")
print(f"  PF              = {pf:.4f}")
print(f"  THD             = {thd*100:.2f}%")
print(f"  Duty range      = {sim_cl['duty'][mask].min():.4f} .. {sim_cl['duty'][mask].max():.4f}")

if pf > 0.95 and abs(v_o_mean - 400) < 5 and thd < 0.20:
    print()
    print("✅  Closed-loop DCM PFC PROVEN: PF > 0.95, V_o regulated, THD < 20%")
else:
    print()
    print("⚠️   Performance off-target — revisit compensator design.")


## 9. Summary

DCM boost PFC delivers:

- **Natural input-current shaping** by physics — the inductor current
  envelope naturally tracks $v_g$ if $D$ is held roughly constant.
- **First-order voltage-loop plant** $G_{vd}(s) = K/(1 + sRC/2)$ — a
  simple PI with $\tau_z = RC/2$ cancels the pole, leaving a pure
  integrator at the desired crossover.
- **No RHP zero, no current loop, no multiplier** — the simplest PFC
  architecture.
- **PF > 0.99 across low/mid line**, degrading to ~0.97 at high line
  due to cusp distortion.

The price: large peak inductor current (~5-6 A at 100 W) and the
sharp triangle waveform creates EMI that the input filter has to
absorb.

**Next**: `03_boost_pfc_ccm.ipynb` does the same exercise with a
CCM design — larger $L$, lower peak current, but a **two-loop**
controller with multiplier between the loops.

**Suggested exercises**

1. Vary the load (change `p.P_o` to 50 W). Re-simulate; does the
   loop hold $V_o$? Compute PF / THD at half load.
2. Step the line voltage during the simulation. The slow voltage
   loop will take ~200 ms to re-regulate. Plot the transient.
3. Reduce $C$ to 47 µF. The output ripple should grow to ~30 V
   pk-pk (predicted by $I_o / (\pi f_{line} C)$).
4. Increase the compensator crossover to 30 Hz. Watch the input
   current shape get distorted at 100 Hz — the loop is now
   "correcting" the 2·f_line ripple instead of leaving it alone.
